In [110]:
import gymnasium as gym, torch, numpy as np, mujoco
import random
import torch.nn as nn
print(gym.__version__, torch.__version__, mujoco.__version__)

0.29.1 2.14.0+cu130 3.14.0


In [111]:
for env_id in ["HalfCheetah-v4", "Walker2d-v4"]:
    env = gym.make(env_id)
    state, _ = env.reset(seed=0)
    print(env_id, "state:", env.observation_space.shape, "action:", env.action_space.shape)
    for _ in range(10):
        state, reward, terminated, truncated, _ = env.step(env.action_space.sample())
    env.close()
    

HalfCheetah-v4 state: (17,) action: (6,)
Walker2d-v4 state: (17,) action: (6,)


In [112]:
env = gym.make("HalfCheetah-v4", render_mode="rgb_array")
env.reset(seed=0)
print(env.render().shape)  # expect something like (480, 480, 3)

(480, 480, 3)


Seeding: we fix Python, NumPy and PyTorch random generators so each run is reproducible. Different seeds give different network initializations and environment starts, letting us measure variability across runs.

In [113]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [114]:
set_seed(0)


In [115]:
print(np.random.rand(), torch.rand(1))

0.5488135039273248 tensor([0.4963])


In [116]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.mean_net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, action_dim)
        )
        self.log_std = nn.Parameter(torch.full((action_dim,), -0.5))
    
    def dist(self, state):
        mean = self.mean_net(state)
        std = self.log_std.exp()
        dist = torch.distributions.Normal(mean, std)
        return dist

    def log_prob(self, dist, action):
        return dist.log_prob(action).sum(-1)
    
    # def forward(self, state):
    #     mean = self.mean_net(state)
    #     dist = torch.distributions.Normal(mean, self.log_std.exp())
    #     return dist

In [117]:
actor = Actor(17, 6)
state = torch.randn(5, 17)
d = actor.dist(state)
action = d.sample()
logp = actor.log_prob(d, action)
print(action.shape, logp.shape)
print(actor.log_std.exp())
print(len(list(actor.parameters())))

torch.Size([5, 6]) torch.Size([5])
tensor([0.6065, 0.6065, 0.6065, 0.6065, 0.6065, 0.6065],
       grad_fn=<ExpBackward0>)
7


In [118]:
class Critic(nn.Module):
    def __init__(self, state_dim):
        super().__init__()
        self.v_net = nn.Sequential(
            nn.Linear(state_dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, state):
        return self.v_net(state).squeeze(-1)

In [119]:
critic = Critic(17)
value = critic(torch.randn(5, 17))
print(value.shape, len(list(critic.parameters())))

torch.Size([5]) 6


In [120]:
class RunningMeanStd:
    def __init__(self, shape):
        self.mean  = np.zeros(shape)
        self.var   = np.ones(shape)
        self.count = 1e-4  

    def update(self, x):
        # x has shape (batch, state_dim)
        # steps below
        batch_mean = x.mean(axis=0)
        batch_var = x.var(axis=0)
        batch_count = x.shape[0]
        delta = batch_mean - self.mean
        total = self.count + batch_count
        new_mean = self.mean + delta * batch_count / total
        M2 = self.var * self.count + batch_var * batch_count + delta**2 * self.count * batch_count / total
        self.mean = new_mean
        self.var = M2 / total
        self.count = total

    def normalize(self, x):
        return np.clip((x - self.mean) / np.sqrt(self.var + 1e-8), -10, 10)

In [121]:
rms = RunningMeanStd((17,))
data = np.random.normal(5, 3, size=(10000, 17))
for i in range(0, 10000, 100):          # feed in chunks, like real training
    rms.update(data[i:i+100])
print(rms.mean[:3], np.sqrt(rms.var[:3]))
n = rms.normalize(data)
print(n.mean(), n.std())

[4.9787411  4.98268914 5.0427004 ] [2.98817081 2.96641892 3.00812517]
1.6721367320743037e-08 0.9999999899051148


In [122]:
def compute_gae(reward, value, next_value, terminated, ended, gamma=0.99, lam=0.95):
    N = len(reward)
    adv = np.zeros(N, dtype=np.float32)
    last_adv = 0.0
    for t in reversed(range(N)):
        delta = reward[t] + gamma * (1 - terminated[t]) * next_value[t] - value[t]
        last_adv = delta + gamma * lam * (1 - ended[t]) * last_adv
        adv[t] = last_adv
    ret = adv + value            # adv + value
    return adv, ret

In [123]:

class RolloutBuffer:
    def __init__(self, size, state_dim, action_dim):
        self.state      = np.zeros((size, state_dim), dtype=np.float32)
        self.action     = np.zeros((size, action_dim), dtype=np.float32)
        self.logp       = np.zeros(size, dtype=np.float32)
        self.reward     = np.zeros(size, dtype=np.float32)
        self.value      = np.zeros(size, dtype=np.float32)
        self.next_value = np.zeros(size, dtype=np.float32)
        self.terminated = np.zeros(size, dtype=np.float32)
        self.ended      = np.zeros(size, dtype=np.float32)
        self.ptr = 0
        self.size = size

    def store(self, state, action, logp, reward, value, next_value, terminated, ended):
        self.state[self.ptr]      = state
        self.action[self.ptr]     = action
        self.logp[self.ptr]       = logp
        self.reward[self.ptr]     = reward
        self.value[self.ptr]      = value
        self.next_value[self.ptr] = next_value
        self.terminated[self.ptr] = terminated
        self.ended[self.ptr]      = ended
        self.ptr += 1

    def get(self, gamma=0.99, lam=0.95):
        adv, ret = compute_gae(self.reward, self.value, self.next_value,
                            self.terminated, self.ended, gamma, lam)
        self.ptr = 0
        t = lambda x: torch.as_tensor(x, dtype=torch.float32)
        return {
            "state":  t(self.state),
            "action": t(self.action),
            "logp":   t(self.logp),
            "adv":    t(adv),
            "ret":    t(ret),
        }

In [124]:
z = np.zeros(3); r = np.ones(3)
e = np.array([0., 1., 0.]); nv = np.array([0., 5., 0.])
print(compute_gae(r, z, z, z, z, 1.0, 1.0)[0])    # expect [3. 2. 1.]
print(compute_gae(r, z, z, e, e, 1.0, 1.0)[0])    # expect [2. 1. 1.]  (terminated at t=1)
print(compute_gae(r, z, nv, z, e, 1.0, 1.0)[0])   # expect [7. 6. 1.]  (truncated at t=1)

[3. 2. 1.]
[2. 1. 1.]
[7. 6. 1.]


In [125]:
buf = RolloutBuffer(4, 17, 6)
for _ in range(4):
    buf.store(np.random.randn(17), np.random.randn(6), 0.0, 1.0, 0.0, 0.0, 0.0, 0.0)
print(buf.ptr)                                    # expect 4 (full)
data = buf.get()
print({k: v.shape for k, v in data.items()})      # state [4,17], action [4,6], others [4]
print(buf.ptr)                                    # expect 0 (reset)

4
{'state': torch.Size([4, 17]), 'action': torch.Size([4, 6]), 'logp': torch.Size([4]), 'adv': torch.Size([4]), 'ret': torch.Size([4])}
0


In [126]:
import pandas as pd
import os

class Logger:
    def __init__(self):
        self.records = []

    def log(self, step, ep_return, ep_length):
        self.records.append({"step": step, "return": ep_return, "length": ep_length})

    def save(self, path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        pd.DataFrame(self.records).to_csv(path, index=False)

In [127]:
def update_critic(critic, v_opt, data, iters=80):
    for _ in range(iters):
        value = critic(data["state"])
        loss = ((value - data["ret"]) ** 2).mean()
        v_opt.zero_grad()
        loss.backward()
        v_opt.step()
    return loss.item()

In [128]:
def vpg_update(actor, critic, pi_opt, v_opt, data):
    state, action, adv = data["state"], data["action"], data["adv"]

    # 1. normalize advantages → mean 0, std 1
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)

    # 2. log π(a|s) under the CURRENT actor (with gradients on)
    d = actor.dist(state)
    logp = actor.log_prob(d, action)

    # 3. policy loss: minimizing -Σ log π · Â  =  ascent on Ĝ
    loss_pi = -(logp * adv).mean()

    # 4. one gradient step on the actor
    pi_opt.zero_grad()
    loss_pi.backward()
    pi_opt.step()

    # 5. fit the critic
    loss_v = update_critic(critic, v_opt, data)
    return loss_pi.item(), loss_v


In [129]:
from torch.nn.utils import parameters_to_vector, vector_to_parameters

def flat_grad(y, params, **kw):
    return torch.cat([g.reshape(-1) for g in torch.autograd.grad(y, params, **kw)])

def conjugate_gradient(Fvp, g, iters=10):
    # solves F x = g using only products F·v  (never forms F)
    x = torch.zeros_like(g)
    r, p = g.clone(), g.clone()
    rr = r @ r
    for _ in range(iters):
        Fp = Fvp(p)
        alpha = rr / (p @ Fp + 1e-8)
        x += alpha * p
        r -= alpha * Fp
        new_rr = r @ r
        if new_rr < 1e-10:
            break
        p = r + (new_rr / rr) * p
        rr = new_rr
    return x

def trpo_update(actor, critic, v_opt, data, delta=0.01, damping=0.1, backtrack_iters=10, backtrack_coef=0.8):
    state, action, logp_old, adv = data["state"], data["action"], data["logp"], data["adv"]
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    params = list(actor.parameters())
    with torch.no_grad():
        old_dist = actor.dist(state)                      # π_θk

    def surrogate():                                      # L_θk(θ) = E[ w · Â ]
        ratio = torch.exp(actor.log_prob(actor.dist(state), action) - logp_old)
        return (ratio * adv).mean()

    def mean_kl():                                        # D̄_KL, same Hessian (F) either direction
        return torch.distributions.kl_divergence(old_dist, actor.dist(state)).sum(-1).mean()

    def Fvp(v):                                           # F·v = ∇(∇KL · v), plus damping
        grad_kl = flat_grad(mean_kl(), params, create_graph=True)
        return flat_grad(grad_kl @ v, params) + damping * v

    # 1. g = ∇L at θk
    loss_old = surrogate()
    g = flat_grad(loss_old, params)

    # 2. v ≈ F⁻¹ g via conjugate gradient
    v = conjugate_gradient(Fvp, g)

    # 3. step to the trust-region boundary: sqrt(2δ / gᵀv) · v
    full_step = torch.sqrt(2 * delta / (g @ v + 1e-8)) * v

    # 4. backtracking line search on α: keep KL ≤ δ and require improvement
    theta_k = parameters_to_vector(params).detach()
    with torch.no_grad():
        for i in range(backtrack_iters):
            vector_to_parameters(theta_k + backtrack_coef**i * full_step, params)
            if mean_kl() <= delta and surrogate() > loss_old:
                break
        else:
            vector_to_parameters(theta_k, params)         # no acceptable step → stay at θk

    # 5. fit the critic
    loss_v = update_critic(critic, v_opt, data)
    return -loss_old.item(), loss_v

In [130]:
def ppo_update(actor, critic, pi_opt, v_opt, data, clip=0.2, epochs=10, minibatch=64, target_kl=0.015):
    state, action, logp_old, adv = data["state"], data["action"], data["logp"], data["adv"]
    adv = (adv - adv.mean()) / (adv.std() + 1e-8)
    N = len(adv)

    for _ in range(epochs):
        idx = torch.randperm(N)
        for start in range(0, N, minibatch):
            mb = idx[start:start + minibatch]
            logp = actor.log_prob(actor.dist(state[mb]), action[mb])
            ratio = torch.exp(logp - logp_old[mb])                          # w_θk(θ, s, a)
            clipped = ratio.clamp(1 - clip, 1 + clip)
            loss_pi = -torch.min(ratio * adv[mb], clipped * adv[mb]).mean()  # -L^ε
            pi_opt.zero_grad()
            loss_pi.backward()
            pi_opt.step()

        # early stop once the policy has drifted too far from π_θk
        with torch.no_grad():
            approx_kl = (logp_old - actor.log_prob(actor.dist(state), action)).mean().item()
        if approx_kl > 1.5 * target_kl:
            break

    loss_v = update_critic(critic, v_opt, data)
    return loss_pi.item(), loss_v

In [131]:
def train(algo, env_id, seed, total_steps=1_000_000, batch=2048):
    set_seed(seed)
    env = gym.make(env_id)
    state, _ = env.reset(seed=seed)
    state_dim, action_dim = env.observation_space.shape[0], env.action_space.shape[0]

    actor, critic = Actor(state_dim, action_dim), Critic(state_dim)
    pi_opt = torch.optim.Adam(actor.parameters(), lr=3e-4)
    v_opt  = torch.optim.Adam(critic.parameters(), lr=1e-3)
    state_rms, buf, logger = RunningMeanStd((state_dim,)), RolloutBuffer(batch, state_dim, action_dim), Logger()
    T = lambda x: torch.as_tensor(x, dtype=torch.float32)

    ep_ret, ep_len, step = 0.0, 0, 0
    while step < total_steps:
        # ---- COLLECT ----
        for _ in range(batch):
            state_rms.update(state[None])
            s = state_rms.normalize(state)
            with torch.no_grad():
                d = actor.dist(T(s)); action = d.sample()
                logp = actor.log_prob(d, action); value = critic(T(s))
            next_state, reward, terminated, truncated, _ = env.step(np.clip(action.numpy(), -1, 1))
            with torch.no_grad():
                next_value = critic(T(state_rms.normalize(next_state)))
            ended = terminated or truncated
            buf.store(s, action.numpy(), logp.item(), reward, value.item(), next_value.item(),
                      float(terminated), float(ended))

            ep_ret += reward; ep_len += 1; step += 1
            if ended:
                logger.log(step, ep_ret, ep_len)
                ep_ret, ep_len = 0.0, 0
                state, _ = env.reset()
            else:
                state = next_state

        # ---- PROCESS + UPDATE ----
        data = buf.get()
        if algo == "vpg":
            vpg_update(actor, critic, pi_opt, v_opt, data)
        elif algo == "trpo":
            trpo_update(actor, critic, v_opt, data)
        elif algo == "ppo":
            ppo_update(actor, critic, pi_opt, v_opt, data)

        if logger.records:
            print(f"step {step:>8} | last return {logger.records[-1]['return']:8.1f}")

    logger.save(f"results/{algo}_{env_id}_seed{seed}.csv")
    torch.save({"actor": actor.state_dict(), "mean": state_rms.mean,
                "var": state_rms.var, "count": state_rms.count},
               f"results/{algo}_{env_id}_seed{seed}.pt")
    env.close()
    return logger

In [134]:
import time
t0 = time.time()
lg = train("ppo", "HalfCheetah-v4", seed=0, total_steps=50_000)
print(f"took {time.time()-t0:.0f}s")

step     2048 | last return   -165.1
step     4096 | last return   -234.2
step     6144 | last return   -226.0
step     8192 | last return   -152.3
step    10240 | last return    -69.3
step    12288 | last return    -79.7
step    14336 | last return    -63.6
step    16384 | last return   -196.8
step    18432 | last return    -73.9
step    20480 | last return    -71.4
step    22528 | last return    -61.4
step    24576 | last return     15.1
step    26624 | last return     35.7
step    28672 | last return   -165.4
step    30720 | last return     54.1
step    32768 | last return     55.2
step    34816 | last return   -230.8
step    36864 | last return    153.5
step    38912 | last return    100.1
step    40960 | last return    221.5
step    43008 | last return    151.5
step    45056 | last return    241.4
step    47104 | last return    148.6
step    49152 | last return    236.0
step    51200 | last return    432.9
took 31s
